# Arm Sequence Tuning

Tune `safe_home -> arm_down -> base push -> grab -> carry`, writing only the `arm` section of `empirical_parameters.json`.

In [ ]:
from __future__ import print_function

import copy
import os
import sys
import traceback

def find_project_root():
    current = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isfile(os.path.join(current, 'config.json')):
            return current
        parent = os.path.dirname(current)
        if parent == current: break
        current = parent
    raise RuntimeError('config.json not found')

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path: sys.path.insert(0, PROJECT_ROOT)

import ipywidgets as widgets
from IPython.display import display
from demo_core import load_config, load_empirical_parameters, save_empirical_parameters
from demo_core.robot_control import ArmController, BaseController

empirical = load_empirical_parameters()
output = widgets.Output(layout={'border': '1px solid #ccc', 'height': '420px', 'overflow_y': 'auto'})
states = ['safe_home', 'arm_down', 'grab', 'carry', 'release']
selected = widgets.Dropdown(options=states, value='safe_home', description='state')
dry_run_arm = widgets.Checkbox(value=True, description='dry_run_arm')
arm_speed = widgets.IntText(value=int(empirical['arm']['speed']), description='arm_speed')
settle = widgets.FloatText(value=float(empirical['arm'].get('servo_settle_seconds', 0.35)), description='settle')
order = widgets.Text(value='5,4,3,2,1', description='order')
servos = {name: widgets.IntText(description=name) for name in ('s1', 's2', 's3', 's4', 's5')}

def parse_order():
    values = [int(value.strip()) for value in order.value.split(',') if value.strip()]
    if sorted(values) != [1, 2, 3, 4, 5]: raise ValueError('order must contain 1,2,3,4,5 once')
    return values

def load_state_into_ui(state):
    definition = empirical['arm']['poses'][state]
    for name, widget in servos.items(): widget.value = int(definition['angles'][name])
    order.value = ','.join(str(value) for value in definition.get('order', [5,4,3,2,1]))

def runtime_config(base_real=False):
    return load_config(overrides={'runtime': {'dry_run': {'camera': True, 'base': not base_real, 'arm': bool(dry_run_arm.value)}}})

def apply_ui_pose():
    state = selected.value
    config = runtime_config()
    config.data['arm']['speed'] = int(arm_speed.value)
    config.data['arm']['servo_settle_seconds'] = float(settle.value)
    config.data['arm']['poses'][state]['order'] = parse_order()
    pose = {name: int(widget.value) for name, widget in servos.items()}
    ArmController(config).pose(state, pose)
    return {'applied': state, 'pose': pose, 'disk_write': False}

def save_selected_pose():
    state = selected.value
    empirical['arm']['speed'] = int(arm_speed.value)
    empirical['arm']['servo_settle_seconds'] = float(settle.value)
    empirical['arm']['poses'][state]['angles'] = {name: int(widget.value) for name, widget in servos.items()}
    empirical['arm']['poses'][state]['order'] = parse_order()
    path = save_empirical_parameters(empirical)
    return {'saved': state, 'path': path}

def show_saved_pose(state):
    config = runtime_config()
    ArmController(config).pose(state)
    return {'shown': state, 'disk_write': False, 'ui_changed': False}

def reload_parameters():
    global empirical
    empirical = load_empirical_parameters()
    arm_speed.value = int(empirical['arm']['speed'])
    settle.value = float(empirical['arm'].get('servo_settle_seconds', 0.35))
    load_state_into_ui(selected.value)
    return 'reloaded empirical_parameters.json'

def push_base():
    fresh = load_empirical_parameters()
    push = fresh['arm']['push']
    speed = float(push['speed']); seconds = float(push['seconds'])
    if speed <= 0 or seconds <= 0: return 'push disabled by zero speed/seconds'
    BaseController(runtime_config(base_real=True)).pulse('forward', speed, seconds, 'arm_tuning_push')
    return {'speed': speed, 'seconds': seconds}

def pickup_sequence():
    config = runtime_config()
    arm = ArmController(config)
    arm.pose('safe_home'); arm.pose('arm_down'); push_base(); arm.pose('grab'); arm.pose('carry')
    return True

def release_sequence():
    arm = ArmController(runtime_config()); arm.pose('release'); arm.pose('safe_home'); return True

def logged(fn):
    def wrapped(_=None):
        with output:
            try: print('\n>>> {}\n{}'.format(fn.__name__, fn()))
            except Exception as exc: print('[error] {}'.format(exc)); traceback.print_exc()
    return wrapped

def select_state(change):
    if change.get('name') == 'value': load_state_into_ui(change['new'])
selected.observe(select_state)
load_state_into_ui('safe_home')
apply_button = widgets.Button(description='Apply UI Pose', button_style='info'); apply_button.on_click(logged(apply_ui_pose))
save_button = widgets.Button(description='Save Selected', button_style='warning'); save_button.on_click(logged(save_selected_pose))
reload_button = widgets.Button(description='Reload'); reload_button.on_click(logged(reload_parameters))
push_button = widgets.Button(description='Push Base'); push_button.on_click(logged(push_base))
pickup_button = widgets.Button(description='Pickup Sequence', button_style='success'); pickup_button.on_click(logged(pickup_sequence))
release_button = widgets.Button(description='Release + Safe'); release_button.on_click(logged(release_sequence))
show_buttons = []
for state in states:
    button = widgets.Button(description='Show ' + state, layout=widgets.Layout(width='145px'))
    button.on_click(logged(lambda value=state: show_saved_pose(value)))
    show_buttons.append(button)
display(widgets.VBox([widgets.HBox([dry_run_arm, selected]), widgets.HBox([arm_speed, settle, order]), widgets.HBox(list(servos.values())), widgets.HBox([apply_button, save_button, reload_button, push_button]), widgets.HBox(show_buttons), widgets.HBox([pickup_button, release_button]), output]))
print('[arm-tune] ready; save writes only empirical_parameters.json arm values')
